# BERT vs. LSTM: What Contextual Embeddings Buy You

**COMP 395 — Deep Learning | Transformers Unit — Day 2 Follow-Up**

You've already classified IMDB reviews using a GloVe + LSTM pipeline (Lab 6). Today you'll run the same task using **frozen DistilBERT embeddings** and a linear classifier — then compare.

The question: **does a better representation make the classifier less important?**

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import time

# Reproducibility
torch.manual_seed(395)
np.random.seed(395)

# Device selection (same pattern as Lab 6)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')

# Install if needed
try:
    from transformers import DistilBertTokenizer, DistilBertModel
    from datasets import load_dataset
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'transformers', 'datasets', '--break-system-packages', '-q'])
    from transformers import DistilBertTokenizer, DistilBertModel
    from datasets import load_dataset

print('Libraries loaded!')

---

## Part 1: Load IMDB Dataset

We'll use HuggingFace `datasets` to load IMDB — same reviews you classified in Lab 6, loaded differently.

In [ ]:
# Load IMDB
dataset = load_dataset('imdb')

print(f"Train: {len(dataset['train'])} reviews")
print(f"Test:  {len(dataset['test'])} reviews")
print(f"\nExample review (first 200 chars):")
print(f"  Text: {dataset['train'][0]['text'][:200]}...")
print(f"  Label: {dataset['train'][0]['label']} ({'positive' if dataset['train'][0]['label'] == 1 else 'negative'})")

In [ ]:
# Subsample for speed — 5,000 train, 2,000 test
# (Full dataset works too, just takes longer for embedding extraction)
n_train = 5000
n_test = 2000

train_texts = dataset['train']['text'][:n_train]
train_labels = dataset['train']['label'][:n_train]
test_texts = dataset['test']['text'][:n_test]
test_labels = dataset['test']['label'][:n_test]

print(f"Using {n_train} train, {n_test} test reviews")
print(f"Label balance (train): {sum(train_labels)}/{n_train} positive")

---

## Part 2: Extract Frozen DistilBERT Embeddings

Here's the key idea: we use DistilBERT as a **frozen feature extractor**. No gradients flow into BERT — we just ask it to produce a 768-dimensional contextual embedding for each review, then train a classifier on those vectors.

DistilBERT has everything you learned about on Day 2:
- **6 layers × 12 attention heads** ($d_k = 64$ per head)
- **Learned positional embeddings** (not sinusoidal — same idea, different implementation)
- **Self-attention** within each review: every word attends to every other word

We'll use the **[CLS] token** embedding as the representation for the whole review. This is a special token prepended to every input — after passing through all 6 transformer layers, it contains a summary of the full sequence.

In [ ]:
# Load DistilBERT tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')
model.eval()  # No dropout
model.to(device)

# Count parameters (all frozen — we won't train any of these)
total_params = sum(p.numel() for p in model.parameters())
print(f"DistilBERT parameters: {total_params:,} (all frozen)")
print(f"Architecture: 6 layers, 12 heads, d_model=768, d_k=64")
print(f"\nThis is everything from Day 2, pretrained on BookCorpus + Wikipedia.")

In [ ]:
def extract_embeddings(texts, tokenizer, model, device, batch_size=32, max_length=512):
    """
    Extract [CLS] embeddings from frozen DistilBERT.
    
    Args:
        texts: list of strings
        tokenizer: DistilBERT tokenizer
        model: frozen DistilBERT model
        device: torch device
        batch_size: how many reviews to process at once
        max_length: max tokens per review (DistilBERT max is 512)
    
    Returns:
        embeddings: tensor of shape (len(texts), 768)
    """
    all_embeddings = []
    
    with torch.no_grad():  # No gradients — BERT is frozen
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            
            # Tokenize
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            ).to(device)
            
            # Forward pass through DistilBERT
            outputs = model(**encoded)
            
            # Extract [CLS] token embedding (position 0)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (batch, 768)
            all_embeddings.append(cls_embeddings.cpu())
            
            if (i // batch_size) % 20 == 0:
                print(f"  Processed {i+len(batch_texts)}/{len(texts)} reviews...")
    
    return torch.cat(all_embeddings, dim=0)

In [ ]:
# Extract embeddings (this takes a few minutes)
print("Extracting train embeddings...")
start = time.time()
train_embeddings = extract_embeddings(train_texts, tokenizer, model, device)
train_time = time.time() - start

print(f"\nExtracting test embeddings...")
test_embeddings = extract_embeddings(test_texts, tokenizer, model, device)

print(f"\nTrain embeddings shape: {train_embeddings.shape}")
print(f"Test embeddings shape:  {test_embeddings.shape}")
print(f"Extraction time: {train_time:.1f}s")
print(f"\nEach review is now a single 768-dimensional vector")
print(f"that encodes contextual meaning — thanks to self-attention.")

---

## Part 3: Train a Linear Classifier

Now the surprising part: we train **only a linear layer** on these embeddings. No RNN, no LSTM, no attention — just a 768 → 2 linear projection.

If DistilBERT's contextual embeddings are good enough, this should work.

In [ ]:
# Prepare data
train_labels_t = torch.tensor(train_labels, dtype=torch.long)
test_labels_t = torch.tensor(test_labels, dtype=torch.long)

train_dataset = TensorDataset(train_embeddings, train_labels_t)
test_dataset = TensorDataset(test_embeddings, test_labels_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
# The simplest possible classifier: one linear layer
class LinearClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        return self.linear(x)

classifier = LinearClassifier(input_dim=768, num_classes=2)

# Count trainable parameters
trainable_params = sum(p.numel() for p in classifier.parameters())
print(f"Trainable parameters: {trainable_params:,}")
print(f"(That's 768 × 2 weights + 2 biases = {768*2 + 2})")
print(f"\nCompare to your Lab 6 LSTM: probably 2-5 MILLION trainable parameters.")

In [ ]:
# Train
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_losses = []
test_accs = []

for epoch in range(20):
    # Train
    classifier.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = classifier(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))
    
    # Evaluate
    classifier.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            preds = classifier(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += len(y_batch)
    acc = correct / total
    test_accs.append(acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}: loss={train_losses[-1]:.4f}, test_acc={acc:.4f}")

print(f"\nFinal test accuracy: {test_accs[-1]:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(test_accs)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy')
axes[1].axhline(y=0.87, color='red', linestyle='--', alpha=0.5, label='Typical LSTM (~87%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Frozen DistilBERT + Linear Classifier on IMDB', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 4: Head-to-Head Comparison

Fill in your Lab 6 results and compare.

In [ ]:
# ── Fill in your Lab 6 results ──
lstm_accuracy = ...     # TODO: your Lab 6 test accuracy (e.g., 0.87)
lstm_params = ...       # TODO: your Lab 6 total trainable parameters (e.g., 2_500_000)
lstm_embedding = "GloVe-50d (static)"  # or nn.Embedding trained from scratch

bert_accuracy = test_accs[-1]
bert_params = trainable_params  # just the linear layer

print("=" * 65)
print(f"{'':>30s}  {'LSTM (Lab 6)':>14s}  {'BERT+Linear':>14s}")
print("-" * 65)
print(f"{'Embedding type':>30s}  {lstm_embedding:>14s}  {'DistilBERT':>14s}")
print(f"{'Embedding dimension':>30s}  {'50':>14s}  {'768':>14s}")
print(f"{'Context':>30s}  {'sequential':>14s}  {'self-attention':>14s}")
print(f"{'Trainable parameters':>30s}  {str(lstm_params):>14s}  {str(bert_params):>14s}")
print(f"{'Test accuracy':>30s}  {str(lstm_accuracy):>14s}  {bert_accuracy:>14.4f}")
print("=" * 65)

### ✏️ Prediction Check

**Answer in this cell:**

1. Did frozen DistilBERT + Linear beat your trained LSTM? By how much?

   *Your answer:*

2. DistilBERT has ~1,500 trainable parameters vs. your LSTM's ~2-5 million. How is this possible?

   *Your answer:*

3. What does this tell you about the relative importance of **representation quality** vs. **classifier complexity**?

   *Your answer:*

---

## Part 5: Peek Inside — What Makes BERT's Embeddings Better?

Let's look at what "contextual" actually means in practice.

In [ ]:
# Same word, different contexts
sentences = [
    "The movie was not good at all.",
    "This was a really good movie!",
    "I went to the bank to deposit money.",
    "We sat on the river bank and watched the sunset.",
]

print("Extracting embeddings for the same words in different contexts...\n")

with torch.no_grad():
    for sent in sentences:
        encoded = tokenizer(sent, return_tensors='pt').to(device)
        outputs = model(**encoded)
        
        # Get token-level embeddings (not just [CLS])
        tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
        embeddings = outputs.last_hidden_state[0].cpu()  # (seq_len, 768)
        
        print(f'"{sent}"')
        print(f"  Tokens: {tokens}")
        print(f"  Embedding shape: {embeddings.shape}")
        print()

In [ ]:
# Compare "good" in positive vs negative context
sent_neg = "The movie was not good at all."
sent_pos = "This was a really good movie!"

with torch.no_grad():
    enc_neg = tokenizer(sent_neg, return_tensors='pt').to(device)
    enc_pos = tokenizer(sent_pos, return_tensors='pt').to(device)
    
    emb_neg = model(**enc_neg).last_hidden_state[0].cpu()
    emb_pos = model(**enc_pos).last_hidden_state[0].cpu()
    
    tokens_neg = tokenizer.convert_ids_to_tokens(enc_neg['input_ids'][0])
    tokens_pos = tokenizer.convert_ids_to_tokens(enc_pos['input_ids'][0])
    
    # Find "good" in each
    idx_neg = tokens_neg.index('good')
    idx_pos = tokens_pos.index('good')
    
    good_neg = emb_neg[idx_neg]
    good_pos = emb_pos[idx_pos]
    
    cos_sim = F.cosine_similarity(good_neg, good_pos, dim=0)
    
    print(f'"good" in negative context: "{sent_neg}"')
    print(f'"good" in positive context: "{sent_pos}"')
    print(f'\nCosine similarity of "good" embeddings: {cos_sim:.4f}')
    print(f'\nWith GloVe, these would be IDENTICAL (cosine sim = 1.0)')
    print(f'because GloVe assigns the same vector to "good" regardless of context.')
    print(f'\nBERT produces DIFFERENT vectors because self-attention lets')
    print(f'"good" attend to "not" in the first sentence, changing its representation.')

In [ ]:
# Compare "bank" in financial vs river context
sent_fin = "I went to the bank to deposit money."
sent_riv = "We sat on the river bank and watched the sunset."

with torch.no_grad():
    enc_fin = tokenizer(sent_fin, return_tensors='pt').to(device)
    enc_riv = tokenizer(sent_riv, return_tensors='pt').to(device)
    
    emb_fin = model(**enc_fin).last_hidden_state[0].cpu()
    emb_riv = model(**enc_riv).last_hidden_state[0].cpu()
    
    tokens_fin = tokenizer.convert_ids_to_tokens(enc_fin['input_ids'][0])
    tokens_riv = tokenizer.convert_ids_to_tokens(enc_riv['input_ids'][0])
    
    idx_fin = tokens_fin.index('bank')
    idx_riv = tokens_riv.index('bank')
    
    bank_fin = emb_fin[idx_fin]
    bank_riv = emb_riv[idx_riv]
    
    cos_sim_bank = F.cosine_similarity(bank_fin, bank_riv, dim=0)
    
    print(f'"bank" (financial): "{sent_fin}"')
    print(f'"bank" (river):     "{sent_riv}"')
    print(f'\nCosine similarity: {cos_sim_bank:.4f}')
    print(f'\nWith GloVe: identical vectors. BERT: different vectors.')
    print(f'This is the word sense disambiguation problem from Day 1 slides — solved.')

---

## Reflection

1. **The representation matters more than the classifier.** A 1,500-parameter linear layer on BERT embeddings beats a multi-million-parameter LSTM on GloVe embeddings. The "intelligence" lives in the representation, not the classifier.

2. **Static vs. contextual is the key difference.** GloVe gives "good" the same vector whether preceded by "not" or "really." BERT gives it different vectors because self-attention (Day 2) lets each word's representation depend on its context.

3. **This is what all of Day 2 was for.** Self-attention, multi-head attention, positional encoding — they're the machinery that turns static embeddings into contextual ones. BERT is just that machinery, pretrained at scale.

4. **The KV cache cost is real.** DistilBERT processed each review through 6 layers × 12 heads. For every token, it computed and stored K and V vectors. The extraction step was slow because of this. TurboQuant compresses exactly these cached vectors.